# 3D Anomaly Detection Benchmark — Anomaly-ShapeNet + Real3D-AD

**Setup:** enable the **T4 GPU** accelerator (Settings → Accelerator), then *Run All*.

What this notebook does:
1. Installs dependencies and fetches the `ad3d` codebase
2. Downloads **Anomaly-ShapeNet** (HuggingFace, public) and **Real3D-AD** (Google Drive, public)
3. Runs a quick smoke test, then the full **Simple3D-lite** baseline on both datasets
4. Saves per-category results (O-AUROC / P-AUROC / AUPR) to `results/`

Expected wall-clock: data ~15 min · Anomaly-ShapeNet ~30–60 min · Real3D-AD ~2–4 h.
For the full-benchmark runs you can also use *Save Version → Save & Run All* so
everything persists even if you disconnect.

In [ ]:
%pip install -q open3d gdown
import torch
print('GPU available:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

In [ ]:
# ---- get the ad3d codebase -------------------------------------------------
import os, glob

REPO_URL = "https://github.com/YOUR_USERNAME/ad3d.git"   # <-- EDIT: your fork

if not os.path.exists('ad3d'):
    if 'YOUR_USERNAME' not in REPO_URL:
        os.system(f'git clone --depth 1 {REPO_URL}')
    if not os.path.exists('ad3d'):
        # fallback: the folder was uploaded as a Kaggle Dataset (Add Input)
        hits = glob.glob('/kaggle/input/**/run.py', recursive=True)
        assert hits, 'Set REPO_URL to your GitHub fork, or upload the ad3d folder as a Kaggle dataset.'
        os.symlink(os.path.dirname(hits[0]), 'ad3d')
print('code at:', os.path.abspath('ad3d'))

In [ ]:
# ---- download Anomaly-ShapeNet (public, ~2 GB) -----------------------------
import os, glob, zipfile, shutil
from huggingface_hub import snapshot_download

DATA = '/kaggle/temp/ad3d-data' if os.path.exists('/kaggle/temp') else 'data'
os.makedirs(DATA, exist_ok=True)
SHAPENET_ROOT = f'{DATA}/anomaly-shapenet/pcd'

if not os.path.exists(f'{SHAPENET_ROOT}/ashtray0/train'):
    snap = snapshot_download(repo_id='Chopper233/Anomaly-ShapeNet',
                             repo_type='dataset', local_dir=f'{DATA}/hf_asn',
                             max_workers=8)
    hits = glob.glob(f'{DATA}/hf_asn/**/ashtray0/train', recursive=True)
    if not hits:  # data ships as zip(s)
        for z in glob.glob(f'{DATA}/hf_asn/**/*.zip', recursive=True):
            print('extracting', z)
            with zipfile.ZipFile(z) as zf:
                zf.extractall(f'{DATA}/asn_raw')
        hits = glob.glob(f'{DATA}/**/ashtray0/train', recursive=True)
    assert hits, 'Could not locate ashtray0/train after extraction - check the layout.'
    pcd_root = os.path.dirname(os.path.dirname(hits[0]))
    os.makedirs(os.path.dirname(SHAPENET_ROOT), exist_ok=True)
    if pcd_root != SHAPENET_ROOT:
        shutil.move(pcd_root, SHAPENET_ROOT)
    shutil.rmtree(f'{DATA}/hf_asn', ignore_errors=True)
    shutil.rmtree(f'{DATA}/asn_raw', ignore_errors=True)

n_train = len(glob.glob(f'{SHAPENET_ROOT}/*/train/*.pcd'))
print(f'Anomaly-ShapeNet ready at {SHAPENET_ROOT} ({n_train} training clouds)')

In [ ]:
# ---- download Real3D-AD (public Google Drive pcd zip, ~4 GB) ----------------
import os, glob, zipfile, shutil, subprocess

REAL_ROOT = f'{DATA}/real3d-ad'
ZIP = f'{DATA}/real3d-ad-pcd.zip'

if not os.path.exists(f'{REAL_ROOT}/airplane/train'):
    ok = subprocess.run(
        ['gdown', '--fuzzy',
         'https://drive.google.com/file/d/1oM4qjhlIMsQc_wiFIFIVBvuuR8nyk2k0/view?usp=sharing',
         '-O', ZIP]).returncode == 0 and os.path.exists(ZIP)
    if not ok:
        # fallback: you uploaded the zip yourself as a Kaggle dataset
        hits = glob.glob('/kaggle/input/**/real3d*pcd*.zip', recursive=True)
        assert hits, ('gdown failed (Drive quota?). Download real3d-ad-pcd.zip from '
                      'https://github.com/M-3LAB/Real3D-AD manually and Add it as a Kaggle input.')
        ZIP = hits[0]
    with zipfile.ZipFile(ZIP) as zf:
        zf.extractall(f'{DATA}/real3d_raw')
    hits = glob.glob(f'{DATA}/real3d_raw/**/airplane/train', recursive=True)
    assert hits, 'Could not locate airplane/train after extraction.'
    cat_root = os.path.dirname(os.path.dirname(hits[0]))
    os.makedirs(os.path.dirname(REAL_ROOT), exist_ok=True)
    if cat_root != REAL_ROOT:
        shutil.move(cat_root, REAL_ROOT)
    shutil.rmtree(f'{DATA}/real3d_raw', ignore_errors=True)
    os.remove(ZIP)

n_train = len(glob.glob(f'{REAL_ROOT}/*/train/*.pcd'))
print(f'Real3D-AD ready at {REAL_ROOT} ({n_train} training clouds)')

In [ ]:
# ---- smoke test: 2 categories, small settings (a few minutes) ---------------
SMOKE = ('--num-group 512 --group-size 64 --max-nn 60 --n-scales 2 '
         '--points-budget 30000')
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --classes ashtray0,bowl0 {SMOKE} --tag smoke

In [ ]:
# ---- FULL Anomaly-ShapeNet benchmark (40 categories) -----------------------
!python ad3d/run.py --dataset shapenet --data-root "{SHAPENET_ROOT}" --tag simple3dlite

In [ ]:
# ---- FULL Real3D-AD benchmark (12 categories) -------------------------------
!python ad3d/run.py --dataset real3d --data-root "{REAL_ROOT}" --tag simple3dlite

In [ ]:
# ---- summarize --------------------------------------------------------------
import pandas as pd, glob
frames = []
for f in sorted(glob.glob('results/simple3dlite_*.csv')):
    df = pd.read_csv(f, index_col=0)
    df.index.name = 'category'
    df['source'] = f
    frames.append(df)
if frames:
    full = pd.concat(frames)
    means = full.groupby('source')[['o_auroc','p_auroc','p_aupr','p_auroc_pooled','p_aupr_pooled']].mean()
    display(means)
    full.drop(columns='source').to_csv('ad3d_results_summary.csv')
    print('saved ad3d_results_summary.csv')
else:
    print('no full-run results yet')

## Next steps

Open **`ad3d/NOVELTY_ROADMAP.md`** — it maps the open failure modes in these
benchmarks (point-level AUPR, the gemstone/starfish/duck categories, the
360°-train vs single-view-test gap) to the exact functions to modify:

| To change... | Edit... |
|---|---|
| features / descriptors | `src/ad3d/features.py` → `compute_fpfh_ms()` |
| grouping / aggregation | `src/ad3d/features.py` → `lfsa()` |
| memory bank / distance | `src/ad3d/memory.py` |
| scoring rules | `src/ad3d/scoring.py` |
| whole pipeline | `src/ad3d/method.py` → subclass `Simple3DLite` |

Every run writes its full config + per-category results into `results/*.json`,
so tagged comparisons are always reproducible.